# Gestor Tático: teste da análise de vídeo do treino

Protótipo para responder uma pergunta: **a detecção e a identificação dos atletas funcionam no campo e na câmera de um clube de verdade?**

O que sai no fim: distância, m/min, velocidade máxima, sprints e mapa de calor de cada atleta, clipes dos sprints e dos lances marcados, e um relatório HTML.

**Antes de começar**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.
2. Coloque o vídeo numa pasta do seu Google Drive, por exemplo `gestor-tatico-teste/treino.mp4`.
3. Rode as células **na ordem**, de cima para baixo. As que você precisa editar estão marcadas com ✏️.

**Como gravar**: celular deitado num tripé, parado o treino inteiro, na lateral e o mais alto possível, com **o campo inteiro no quadro** (os 4 cantos aparecendo). Coletes numerados ajudam muito na identificação.

**LGPD**: só use vídeo com **autorização por escrito dos responsáveis** (são menores de idade). O vídeo fica no seu Google Drive e é processado nos servidores do Google (Colab). Apague o vídeo e os resultados depois do teste.

In [ ]:
# Instala as bibliotecas (o Colab já tem PyTorch, OpenCV, pandas, scikit-learn e ffmpeg)
!pip install -q ultralytics easyocr

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SEM GPU: ative em Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4).")

## 1. ✏️ Configuração

In [ ]:
# De onde vem o vídeo:
# - False (mais simples): envie o vídeo pelo painel de arquivos (ícone de pasta à esquerda > "content" > ⋮ > Fazer upload)
#   e espere o envio terminar. O resultado fica em /content/resultado: baixe antes de fechar, porque some com a sessão.
# - True: o vídeo fica no seu Google Drive. O Colab pede acesso ao Drive e só funciona com TODAS as permissões marcadas.
USAR_DRIVE = False

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO_ORIGINAL = "/content/drive/MyDrive/gestor-tatico-teste/treino.mp4"
    PASTA_RESULTADO = "/content/drive/MyDrive/gestor-tatico-teste/resultado"
else:
    VIDEO_ORIGINAL = "/content/treino.mp4"
    PASTA_RESULTADO = "/content/resultado"

# Dimensões do campo INTEIRO em metros (meça ou pergunte ao clube). Comprimento = lado paralelo à câmera.
# Referências: society ~40 x 20, futebol 7 ~50 x 30, campo oficial ~100-105 x 64-68.
COMPRIMENTO_M = 40
LARGURA_M = 20

# A câmera ficou parada o vídeo todo? Se ela acompanha a bola (vira de um lado para o outro), use False:
# o teste mede só a detecção e a identificação, sem distância, velocidade nem mapa de calor.
CAMERA_PARADA = True

# Como calibrar: "cantos" (o vídeo mostra os 4 cantos do campo) ou "meio" (o vídeo mostra só o meio-campo:
# a linha do meio, as duas laterais e o círculo central).
CALIBRACAO = "cantos"

# Teste rápido: processa só os primeiros N segundos. Use None para o vídeo inteiro.
FIM_SEGUNDOS = 300

# Grava um vídeo com as caixas e os números lidos (bom para conferir a detecção no teste rápido)
VIDEO_DEBUG = True

# Lista de presença: número do colete -> nome. Pode deixar vazio.
ATLETAS = {
    # 7: "Lucas",
    # 10: "Pedro",
}

# Lances marcados pelo treinador: segundo do vídeo -> nome do atleta (None = lance do time). Pode deixar vazio.
LANCES = {
    # 312.5: "Lucas",
    # 1045: None,
}

# Parâmetros (ajustar depois do 1º teste)
# Sem GPU (o Colab grátis às vezes não tem): FIM_SEGUNDOS = 60, FPS_PROCESSAMENTO = 5, "yolo11s.pt" e IMGSZ = 960.
FPS_PROCESSAMENTO = 10      # quadros por segundo analisados
MODELO_YOLO = "yolo11m.pt"  # baixado sozinho na 1ª execução
IMGSZ = 1280                # resolução da detecção (jogador longe fica pequeno)
CONF_MIN = 0.3
MARGEM_CAMPO_M = 1.5        # aceita jogador até 1,5 m fora da linha
VEL_MAX_VALIDA = 10.0       # m/s; acima disso é erro de rastreio
LIMIAR_SPRINT = 5.0         # m/s (18 km/h); para categorias pequenas, testar 4.0
DURACAO_MIN_SPRINT = 1.0    # s
GAP_MAX = 0.5               # s; buraco máximo sem quebrar a trajetória
OCR_A_CADA_N = 5            # tenta ler o número a cada N aparições do jogador
OCR_ALTURA_MIN_PX = 80      # só lê se o jogador tiver essa altura mínima na imagem
OCR_CONF_MIN = 0.4
MIN_AMOSTRAS_TRACK = 15     # trechos menores que isso são ignorados
N_TIMES = 3                 # grupos de cor: os dois times + árbitro (use 2 num treino sem árbitro)

L, W = COMPRIMENTO_M, LARGURA_M

## 2. Preparar o vídeo

Converte para no máximo 1080p e **30 quadros por segundo constantes** (o celular grava com taxa variável, o que bagunça o tempo e a velocidade), sem áudio. No teste rápido, corta só o trecho do `FIM_SEGUNDOS`.

In [ ]:
import os
import subprocess

import cv2

os.makedirs("/content/trabalho", exist_ok=True)
VIDEO = "/content/trabalho/video.mp4"


def rodar(cmd):
    """Roda o ffmpeg e, se falhar, mostra a mensagem dele (o Colab esconde a saída de erro)."""
    r = subprocess.run(cmd, capture_output=True, text=True)
    assert r.returncode == 0, "ffmpeg falhou:\n" + r.stderr[-2000:]


assert os.path.exists(VIDEO_ORIGINAL), f"Não achei {VIDEO_ORIGINAL}: confira o envio ou o caminho na configuração."

cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", VIDEO_ORIGINAL]
if FIM_SEGUNDOS:
    cmd += ["-t", str(FIM_SEGUNDOS)]
cmd += ["-vf", "scale=-2:'min(1080,ih)'", "-r", "30", "-c:v", "libx264", "-preset", "veryfast",
        "-crf", "20", "-an", VIDEO]
rodar(cmd)

cap = cv2.VideoCapture(VIDEO)
largura_px, altura_px = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duracao_s = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f"Vídeo pronto: {largura_px}x{altura_px}, {duracao_s / 60:.1f} min")

## 3. Calibrar o campo

Clique nos **4 pontos de referência**, na ordem pedida acima da imagem. Isso converte a posição na imagem em metros.

- `CALIBRACAO = "cantos"`: os 4 cantos do campo.
- `CALIBRACAO = "meio"`: onde a linha do meio encontra as duas laterais e os pontos mais à esquerda e mais à direita do círculo central.

Com `CAMERA_PARADA = False` esta etapa é pulada. Se o clique não funcionar, preencha `PONTOS_MANUAIS` na célula seguinte com as coordenadas (em pixels do vídeo) e pule esta.

In [ ]:
import base64
import json

from google.colab.output import eval_js
from IPython.display import Javascript, display

RAIO_CIRCULO = 9.15
if CALIBRACAO == "meio":
    REFERENCIAS = [
        ("ponto onde a LINHA DO MEIO encontra a lateral do lado de lá (fundo)", (L / 2, 0)),
        ("ponto onde a LINHA DO MEIO encontra a lateral do lado da câmera (perto)", (L / 2, W)),
        ("ponto mais à ESQUERDA do círculo central", (L / 2 - RAIO_CIRCULO, W / 2)),
        ("ponto mais à DIREITA do círculo central", (L / 2 + RAIO_CIRCULO, W / 2)),
    ]
else:
    REFERENCIAS = [
        ("canto ESQUERDO do lado de lá (fundo)", (0, 0)),
        ("canto DIREITO do lado de lá (fundo)", (L, 0)),
        ("canto DIREITO do lado da câmera (perto)", (L, W)),
        ("canto ESQUERDO do lado da câmera (perto)", (0, W)),
    ]
CANTOS = [rotulo for rotulo, _ in REFERENCIAS]

cap = cv2.VideoCapture(VIDEO)
cap.set(cv2.CAP_PROP_POS_MSEC, 5000)
ok, QUADRO_CALIBRACAO = cap.read()
cap.release()
assert ok, "Não foi possível ler o vídeo."

escala = 1000 / QUADRO_CALIBRACAO.shape[1]
_, buffer = cv2.imencode(".jpg", cv2.resize(QUADRO_CALIBRACAO, None, fx=escala, fy=escala))

JS_CLIQUES = Javascript("""
async function clicarCantos(b64, rotulos) {
  const msg = document.createElement('p');
  msg.style.font = 'bold 16px sans-serif';
  const canvas = document.createElement('canvas');
  canvas.style.cursor = 'crosshair';
  const img = new Image();
  await new Promise(r => { img.onload = r; img.src = 'data:image/jpeg;base64,' + b64; });
  canvas.width = img.width; canvas.height = img.height;
  const ctx = canvas.getContext('2d');
  ctx.drawImage(img, 0, 0);
  const area = document.querySelector('#output-area');
  area.appendChild(msg); area.appendChild(canvas);
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  const pontos = [];
  msg.textContent = 'Clique no ' + rotulos[0];
  return await new Promise(resolve => {
    canvas.addEventListener('click', e => {
      if (pontos.length >= 4) return;
      const r = canvas.getBoundingClientRect();
      const x = (e.clientX - r.left) * canvas.width / r.width;
      const y = (e.clientY - r.top) * canvas.height / r.height;
      pontos.push([x, y]);
      ctx.fillStyle = 'red';
      ctx.beginPath(); ctx.arc(x, y, 6, 0, 2 * Math.PI); ctx.fill();
      ctx.font = 'bold 18px sans-serif';
      ctx.fillText(String(pontos.length), x + 8, y - 8);
      if (pontos.length < 4) { msg.textContent = 'Clique no ' + rotulos[pontos.length]; }
      else { msg.textContent = 'Pronto! Rode a próxima célula.'; resolve(pontos); }
    });
  });
}
""")
if CAMERA_PARADA:
    display(JS_CLIQUES)
    cliques = eval_js(f"clicarCantos({json.dumps(base64.b64encode(buffer).decode())}, {json.dumps(CANTOS)})")
    PONTOS = [(x / escala, y / escala) for x, y in cliques]
    print("Pontos (pixels do vídeo):", [(round(x), round(y)) for x, y in PONTOS])
else:
    print("CAMERA_PARADA = False: calibração pulada (o teste mede só detecção e identificação).")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PONTOS_MANUAIS = None  # ex.: [(312, 220), (1610, 230), (1890, 960), (40, 950)], na ordem das referências

H = None
if CAMERA_PARADA:
    pontos = PONTOS_MANUAIS or globals().get("PONTOS")
    assert pontos and len(pontos) == 4, "Faltam os 4 pontos: rode a célula anterior ou preencha PONTOS_MANUAIS."
    H = cv2.getPerspectiveTransform(np.float32(pontos), np.float32([coord for _, coord in REFERENCIAS]))

    # Conferência: vista de cima do campo. Se as linhas do campo ficarem retas, a calibração está boa.
    # No modo "meio", a parte do campo que não aparece no vídeo fica preta (normal).
    px_m = 10 if L > 60 else 20
    topo = cv2.warpPerspective(QUADRO_CALIBRACAO, np.diag([px_m, px_m, 1.0]) @ H, (int(L * px_m), int(W * px_m)))
    plt.figure(figsize=(10, 10 * W / L))
    plt.imshow(cv2.cvtColor(topo, cv2.COLOR_BGR2RGB))
    plt.title("Vista de cima: as linhas do campo devem ficar retas")
    plt.axis("off")
    plt.show()
else:
    print("Sem calibração.")

## 4. Processar (parte pesada, usa a GPU)

Detecta e rastreia os jogadores, converte a posição em metros, separa os times pela cor e tenta ler o número do colete. Numa GPU T4, conte alguns minutos para cada 5 minutos de vídeo; o progresso aparece a cada 30 s de vídeo.

In [ ]:
import time
from collections import Counter, defaultdict
from pathlib import Path

import easyocr
import pandas as pd
import yaml
from sklearn.cluster import KMeans
from ultralytics import YOLO
from ultralytics.utils import ROOT

SAIDA = Path("/content/trabalho/saida")
(SAIDA / "recortes").mkdir(parents=True, exist_ok=True)


def projetar(pontos_px):
    """Pixels (N x 2) -> metros no campo (N x 2)."""
    pts = np.asarray(pontos_px, dtype=np.float32).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, H).reshape(-1, 2)


def cor_tronco(frame, caixa):
    """Mediana da cor (LAB) na faixa do tronco: evita cabeça, short e fundo."""
    x1, y1, x2, y2 = caixa
    h, w = y2 - y1, x2 - x1
    recorte = frame[max(int(y1 + 0.2 * h), 0):int(y1 + 0.5 * h), max(int(x1 + 0.25 * w), 0):int(x1 + 0.75 * w)]
    if recorte.size == 0:
        return None
    return np.median(cv2.cvtColor(recorte, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32), axis=0)


def agrupar_times(cores_por_track):
    """Agrupa os tracks pela cor mediana do tronco (LAB completo).
    A luminosidade (L) entra de propósito: camisa preta x branca só se diferencia por ela.
    Ela pesa metade, para uma sombra não separar o mesmo time em dois."""
    ids = [t for t, c in cores_por_track.items() if c]
    if len(ids) < N_TIMES:
        return {t: 0 for t in ids}
    X = np.array([np.median(np.array(cores_por_track[t]), axis=0) for t in ids])
    X[:, 0] *= 0.5
    km = KMeans(n_clusters=N_TIMES, n_init=10, random_state=0).fit(X)
    return dict(zip(ids, km.labels_.tolist()))


leitor_ocr = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))


def ler_numero(frame, caixa):
    """(numero, confianca) do colete, ou None."""
    x1, y1, x2, y2 = caixa.astype(int)
    h = y2 - y1
    recorte = frame[max(int(y1 + 0.15 * h), 0):int(y1 + 0.55 * h), max(x1, 0):x2]
    if recorte.size == 0:
        return None
    f = 128 / max(recorte.shape[0], 1)
    recorte = cv2.resize(recorte, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
    cinza = clahe.apply(cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY))
    melhor = None
    for _, texto, conf in leitor_ocr.readtext(cinza, allowlist="0123456789", detail=1):
        if texto.isdigit() and 1 <= int(texto) <= 99 and conf >= OCR_CONF_MIN:
            if melhor is None or conf > melhor[1]:
                melhor = (int(texto), float(conf))
    return melhor


def votar(leituras):
    """Número mais provável do track, ponderado pela confiança: (numero, votos, proporcao)."""
    if not leituras:
        return None, 0, 0.0
    pesos = Counter()
    for numero, conf in leituras:
        pesos[numero] += conf
    numero, peso = pesos.most_common(1)[0]
    return numero, sum(1 for n, _ in leituras if n == numero), peso / sum(pesos.values())


def cor_do_track(tid):
    return tuple(int(c) for c in np.random.default_rng(tid).integers(64, 255, 3))


# ByteTrack segurando o jogador "perdido" por ~3 s (o padrão é ~1 s), para trocar menos de ID em sobreposição.
# O track_buffer conta quadros processados: 30 quadros a 10 quadros/s = 3 s.
config_tracker = yaml.safe_load(open(ROOT / "cfg/trackers/bytetrack.yaml"))
config_tracker["track_buffer"] = int(3 * FPS_PROCESSAMENTO)
TRACKER = "/content/trabalho/bytetrack_gt.yaml"
with open(TRACKER, "w") as f:
    yaml.safe_dump(config_tracker, f)

modelo = YOLO(MODELO_YOLO)
cap = cv2.VideoCapture(VIDEO)
fps_video = cap.get(cv2.CAP_PROP_FPS) or 30.0
passo = max(1, round(fps_video / FPS_PROCESSAMENTO))
FPS_PROC = fps_video / passo

writer = None
linhas = []                      # uma linha por detecção válida: track, t, x, y (metros)
cores = defaultdict(list)        # track -> amostras de cor do tronco
leituras = defaultdict(list)     # track -> leituras de número
aparicoes = defaultdict(int)
melhor_recorte = {}              # track -> (altura_px, imagem), para a revisão

inicio_proc = time.time()
idx = -1
while cap.grab():                # grab() é barato; só decodifica os quadros usados
    idx += 1
    if idx % passo:
        continue
    t = idx / fps_video
    ok, frame = cap.retrieve()
    if not ok:
        break

    res = modelo.track(frame, persist=True, tracker=TRACKER, classes=[0],
                       conf=CONF_MIN, imgsz=IMGSZ, verbose=False)[0]
    desenhos = []
    if res.boxes.id is not None:
        caixas = res.boxes.xyxy.cpu().numpy()
        ids = res.boxes.id.int().cpu().numpy()
        pes = np.stack([(caixas[:, 0] + caixas[:, 2]) / 2, caixas[:, 3]], axis=1)  # ponto dos pés
        if H is not None:
            campo = projetar(pes)
            m = MARGEM_CAMPO_M
            dentro = (campo[:, 0] >= -m) & (campo[:, 0] <= L + m) & (campo[:, 1] >= -m) & (campo[:, 1] <= W + m)
        else:
            campo = pes                                  # sem calibração: posição em pixels, só para o rastreio
            dentro = np.ones(len(pes), dtype=bool)       # e ninguém é descartado (pais e banco aparecem na revisão)

        for caixa, tid, (xm, ym), no_campo in zip(caixas, ids, campo, dentro):
            if not no_campo:
                continue         # torcida, banco, pais na beira do campo
            tid = int(tid)
            x1, y1, x2, y2 = caixa.astype(int)
            altura = y2 - y1
            linhas.append({"track": tid, "t": round(t, 3), "x": float(xm), "y": float(ym)})
            aparicoes[tid] += 1
            if aparicoes[tid] % 3 == 0:
                cor = cor_tronco(frame, caixa)
                if cor is not None:
                    cores[tid].append(cor)
            if altura >= OCR_ALTURA_MIN_PX and aparicoes[tid] % OCR_A_CADA_N == 0:
                leitura = ler_numero(frame, caixa)
                if leitura:
                    leituras[tid].append(leitura)
            if tid not in melhor_recorte or altura > melhor_recorte[tid][0]:
                melhor_recorte[tid] = (altura, frame[max(y1, 0):y2, max(x1, 0):x2].copy())
            if VIDEO_DEBUG:
                desenhos.append((tid, x1, y1, x2, y2))

    if VIDEO_DEBUG:
        # Desenha só depois de tirar a cor e o número, para não sujar os recortes
        for tid, x1, y1, x2, y2 in desenhos:
            numero = votar(leituras[tid])[0]
            rotulo = f"{tid}" + (f" #{numero}" if numero else "")
            cv2.rectangle(frame, (x1, y1), (x2, y2), cor_do_track(tid), 2)
            cv2.putText(frame, rotulo, (x1, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor_do_track(tid), 2)
        if writer is None:
            writer = cv2.VideoWriter(str(SAIDA / "debug_bruto.mp4"), cv2.VideoWriter_fourcc(*"mp4v"),
                                     FPS_PROC, (frame.shape[1], frame.shape[0]))
        writer.write(frame)

    if idx % (passo * int(FPS_PROC * 30)) == 0:
        print(f"{t / 60:5.1f} min de vídeo · {time.time() - inicio_proc:5.0f} s de processamento · tracks: {len(aparicoes)}")

cap.release()
TEMPO_PROCESSAMENTO_S = time.time() - inicio_proc

if writer:
    writer.release()
    # mp4v não toca no navegador: converte para H.264
    rodar(["ffmpeg", "-y", "-loglevel", "error", "-i", str(SAIDA / "debug_bruto.mp4"),
                    "-c:v", "libx264", "-preset", "veryfast", "-crf", "26", str(SAIDA / "debug.mp4")])
    (SAIDA / "debug_bruto.mp4").unlink()

assert linhas, "Nenhum jogador detectado dentro do campo: confira a calibração."
deteccoes = pd.DataFrame(linhas)
deteccoes.to_csv(SAIDA / "deteccoes.csv", index=False)

times_por_track = agrupar_times(cores)
resumo = []
for tid, grupo in deteccoes.groupby("track"):
    numero, votos, confianca = votar(leituras[tid])
    if len(grupo) < MIN_AMOSTRAS_TRACK:
        atleta = "ignorar"
    elif numero is not None:
        atleta = ATLETAS.get(numero, f"#{numero}")
    else:
        atleta = ""              # precisa de revisão
    resumo.append({"track": tid, "amostras": len(grupo), "inicio_s": grupo["t"].min(), "fim_s": grupo["t"].max(),
                   "time": times_por_track.get(tid, -1), "numero_sugerido": numero, "votos": votos,
                   "confianca": round(confianca, 2), "atleta": atleta})
    if tid in melhor_recorte:
        cv2.imwrite(str(SAIDA / "recortes" / f"track_{tid}.jpg"), melhor_recorte[tid][1])

pd.DataFrame(resumo).to_csv(SAIDA / "identidades.csv", index=False)
pendentes = sum(1 for r in resumo if r["atleta"] == "")
print(f"\nPronto em {TEMPO_PROCESSAMENTO_S / 60:.1f} min: {len(resumo)} tracks, {pendentes} sem identificação.")

### Conferir a detecção

Assista a um trecho do vídeo de debug: cada jogador deve ter uma caixa com um número de track (e, quando lido, `#número do colete`). Repare se algum jogador fica muito tempo sem caixa ou se os números de track trocam a toda hora.

In [ ]:
from IPython.display import HTML

if (SAIDA / "debug.mp4").exists():
    tamanho_mb = (SAIDA / "debug.mp4").stat().st_size / 1e6
    if tamanho_mb < 80:
        dados = base64.b64encode((SAIDA / "debug.mp4").read_bytes()).decode()
        display(HTML(f'<video controls width="900" src="data:video/mp4;base64,{dados}"></video>'))
    else:
        print(f"debug.mp4 tem {tamanho_mb:.0f} MB: grande demais para mostrar aqui. Ele vai para o Drive no fim.")
else:
    print("Sem vídeo de debug (VIDEO_DEBUG = False).")

## 5. Revisar quem é quem

Cada foto é um **track** (um trecho em que o sistema seguiu a mesma pessoa). Em vermelho, os que o sistema não conseguiu identificar pelo número. Um mesmo atleta pode aparecer em vários tracks.

**Cronometre a revisão**: é uma das respostas que o teste precisa dar (meta: menos de 3 a 5 minutos por treino).

In [ ]:
import math

INICIO_REVISAO = time.time()
ident = pd.read_csv(SAIDA / "identidades.csv", dtype={"atleta": str}).fillna({"atleta": ""})
visiveis = ident[ident["atleta"] != "ignorar"].sort_values(["atleta", "inicio_s"])

colunas = 8
fig, eixos = plt.subplots(math.ceil(len(visiveis) / colunas), colunas,
                          figsize=(colunas * 2.2, math.ceil(len(visiveis) / colunas) * 3.4), squeeze=False)
for eixo in eixos.flat:
    eixo.axis("off")
for eixo, (_, r) in zip(eixos.flat, visiveis.iterrows()):
    img = cv2.imread(str(SAIDA / "recortes" / f"track_{r['track']}.jpg"))
    if img is not None:
        eixo.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    nome = r["atleta"] or "???"
    eixo.set_title(f"track {r['track']} · time {r['time']}\n{nome}\n{r['inicio_s'] / 60:.1f}–{r['fim_s'] / 60:.1f} min",
                   fontsize=8, color="red" if not r["atleta"] else "black")
plt.tight_layout()
plt.show()
display(visiveis[["track", "amostras", "inicio_s", "fim_s", "time", "numero_sugerido", "confianca", "atleta"]])

### ✏️ Corrigir

Preencha `CORRECOES` com `track: "Nome"` para identificar ou corrigir, e `track: "ignorar"` para técnico, árbitro, pais e falsos positivos. Os tracks que você não citar ficam como o sistema sugeriu. Pode rodar esta célula e a do relatório quantas vezes quiser, sem reprocessar o vídeo.

In [ ]:
CORRECOES = {
    # 17: "Lucas",
    # 23: "ignorar",
}

ident = pd.read_csv(SAIDA / "identidades.csv", dtype={"atleta": str}).fillna({"atleta": ""})
for track, nome in CORRECOES.items():
    ident.loc[ident["track"] == track, "atleta"] = nome
ident.to_csv(SAIDA / "identidades_revisadas.csv", index=False)

TEMPO_REVISAO_S = time.time() - INICIO_REVISAO
sem_nome = ident[ident["atleta"] == ""]
print(f"Revisão: {TEMPO_REVISAO_S / 60:.1f} min desde que as fotos apareceram.")
print(f"Ainda sem identificação (ficam fora do relatório): {len(sem_nome)} tracks {sem_nome['track'].tolist()}")

## 6. Relatório

In [ ]:
import html
import shutil

from scipy.ndimage import gaussian_filter
from scipy.signal import savgol_filter


def suavizar(valores):
    """Savitzky-Golay com janela de ~1 s: tira o tremido da caixa do detector."""
    janela = int(round(FPS_PROC)) | 1
    janela = min(janela, len(valores) if len(valores) % 2 else len(valores) - 1)
    return savgol_filter(valores, janela, polyorder=2) if janela >= 5 else valores


def segmentos(df):
    """Quebra cada track em trechos contínuos (sem buracos maiores que GAP_MAX)."""
    for _, track in df.sort_values("t").groupby("track"):
        quebra = (track["t"].diff() > GAP_MAX).cumsum()
        for _, s in track.groupby(quebra):
            if len(s) >= 5:
                yield s


def calcular(df_atleta):
    distancia = tempo = vel_max = 0.0
    sprints, posicoes = [], []
    k = max(1, int(round(FPS_PROC * 0.5)))           # média móvel de 0,5 s na velocidade
    for s in segmentos(df_atleta):
        t = s["t"].to_numpy()
        x, y = suavizar(s["x"].to_numpy()), suavizar(s["y"].to_numpy())
        posicoes.append(np.column_stack([x, y]))
        dt = np.diff(t)
        passo_m = np.hypot(np.diff(x), np.diff(y))
        v = passo_m / dt
        validos = v <= VEL_MAX_VALIDA                 # descarta "teletransporte" (troca de ID)
        distancia += passo_m[validos].sum()
        tempo += dt.sum()
        v_suave = pd.Series(np.where(validos, v, np.nan)).rolling(k, center=True, min_periods=1).mean().to_numpy()
        if np.isfinite(v_suave).any():
            vel_max = max(vel_max, float(np.nanmax(v_suave)))
        acima = np.nan_to_num(v_suave) >= LIMIAR_SPRINT
        inicio = None
        for i, a in enumerate(np.append(acima, False)):
            if a and inicio is None:
                inicio = i
            elif not a and inicio is not None:
                if t[i] - t[inicio] >= DURACAO_MIN_SPRINT:
                    sprints.append({"inicio_s": float(t[inicio]), "fim_s": float(t[i]),
                                    "vel_pico": float(np.nanmax(v_suave[inicio:i]))})
                inicio = None
    metricas = {"distancia_m": round(distancia), "tempo_min": round(tempo / 60, 1),
                "m_por_min": round(distancia / tempo * 60, 1) if tempo else 0,
                "vel_max_kmh": round(vel_max * 3.6, 1), "sprints": len(sprints)}
    return metricas, sprints, (np.vstack(posicoes) if posicoes else np.empty((0, 2)))


def mapa_calor(posicoes, caminho, titulo):
    hist, _, _ = np.histogram2d(posicoes[:, 0], posicoes[:, 1], bins=[int(L), int(W)], range=[[0, L], [0, W]])
    fig, ax = plt.subplots(figsize=(8, 8 * W / L))
    ax.imshow(gaussian_filter(hist, sigma=2).T, extent=[0, L, W, 0], cmap="YlOrRd", interpolation="bilinear")
    ax.plot([0, L, L, 0, 0], [0, 0, W, W, 0], color="green", lw=2)
    ax.axvline(L / 2, color="green", lw=1)
    ax.set_title(titulo)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.savefig(caminho, dpi=100, bbox_inches="tight")
    plt.close(fig)


def slug(nome):
    return "".join(c if c.isalnum() else "_" for c in str(nome))


def cortar_clipe(inicio, fim, destino):
    inicio = max(inicio, 0)
    rodar(["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{inicio:.2f}", "-i", VIDEO,
                    "-t", f"{fim - inicio:.2f}", "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
                    "-an", str(destino)])


def tracks_sobrepostos(grupo, tolerancia=2.0):
    """Dois tracks do mesmo atleta ao mesmo tempo = provável erro de identidade."""
    faixas = grupo.groupby("track")["t"].agg(["min", "max"]).sort_values("min").to_numpy()
    fim_max = float("-inf")
    for ini, fim in faixas:
        if ini < fim_max - tolerancia:
            return True
        fim_max = max(fim_max, fim)
    return False


for pasta in ("mapas", "clipes"):
    shutil.rmtree(SAIDA / pasta, ignore_errors=True)
    (SAIDA / pasta).mkdir()

identificados = ident[~ident["atleta"].isin(["", "ignorar"])]
df = deteccoes.merge(identificados[["track", "atleta"]], on="track")
linhas_rel, clipes = [], {}
for atleta, grupo in df.groupby("atleta"):
    if CAMERA_PARADA:
        met, sprints, posicoes = calcular(grupo)
    else:   # sem calibração não há metros: o relatório mostra só a identificação
        met, sprints, posicoes = {}, [], np.empty((0, 2))
    if len(posicoes):
        mapa_calor(posicoes, SAIDA / "mapas" / f"{slug(atleta)}.png", atleta)
    clipes[atleta] = []
    for i, sp in enumerate(sorted(sprints, key=lambda s: -s["vel_pico"])[:2], 1):   # os 2 sprints mais rápidos
        destino = SAIDA / "clipes" / f"{slug(atleta)}_sprint{i}.mp4"
        cortar_clipe(sp["inicio_s"] - 3, sp["fim_s"] + 2, destino)
        clipes[atleta].append(destino)
    linhas_rel.append({"atleta": atleta, **met, "tracks": grupo["track"].nunique(),
                       "alerta_identidade": "sim" if tracks_sobrepostos(grupo) else ""})

for i, (tempo_s, dono) in enumerate(sorted(LANCES.items()), 1):
    if FIM_SEGUNDOS and tempo_s > FIM_SEGUNDOS:
        continue                                     # fora do trecho processado
    dono = dono or "Lances do time"
    destino = SAIDA / "clipes" / f"lance_{i}_{slug(dono)}.mp4"
    cortar_clipe(tempo_s - 10, tempo_s + 5, destino)
    clipes.setdefault(dono, []).append(destino)

assert linhas_rel, "Nenhum atleta identificado: preencha CORRECOES na etapa 5."
tabela = pd.DataFrame(linhas_rel).sort_values("distancia_m" if CAMERA_PARADA else "atleta",
                                              ascending=not CAMERA_PARADA)
tabela.to_csv(SAIDA / "relatorio.csv", index=False)


def videos(nome):
    return "".join(f'<video src="clipes/{c.name}" controls width="320"></video>' for c in clipes.get(nome, []))


def resumo_atleta(r):
    if not CAMERA_PARADA:
        return f"<p>{r['tracks']} trechos rastreados</p>"
    return (f"<p>{r['distancia_m']} m · {r['tempo_min']} min · {r['m_por_min']} m/min · "
            f"máx. {r['vel_max_kmh']} km/h · {r['sprints']} sprints</p>"
            f'<img src="mapas/{slug(r["atleta"])}.png" width="480">')


blocos = [f"<section><h2>{html.escape(r['atleta'])}</h2>{resumo_atleta(r)}{videos(r['atleta'])}</section>"
          for _, r in tabela.iterrows()]
blocos += [f"<section><h2>{html.escape(n)}</h2>{videos(n)}</section>" for n in set(clipes) - set(tabela["atleta"])]
(SAIDA / "relatorio.html").write_text(
    '<!doctype html><meta charset="utf-8"><title>Relatório do treino</title>'
    "<style>body{font-family:sans-serif;max-width:1000px;margin:auto;padding:16px}"
    "section{border-bottom:1px solid #ddd;padding:12px 0}video{margin:4px}</style>"
    f"<h1>Relatório do treino</h1>{tabela.to_html(index=False)}{''.join(blocos)}", encoding="utf-8")

display(tabela)
for _, r in tabela.iterrows():
    caminho = SAIDA / "mapas" / f"{slug(r['atleta'])}.png"
    if caminho.exists():
        display(HTML(f'<img width="420" src="data:image/png;base64,{base64.b64encode(caminho.read_bytes()).decode()}">'))

shutil.copytree(SAIDA, PASTA_RESULTADO, dirs_exist_ok=True)
print(f"Resultado copiado para {PASTA_RESULTADO} (abra relatorio.html depois de baixar a pasta).")

## 7. O que o teste respondeu

Anote estes números. Eles dizem se vale implementar no Gestor Tático.

In [ ]:
validos_ident = ident[ident["atleta"] != "ignorar"]
auto = validos_ident["numero_sugerido"].notna().sum()
min_video = (FIM_SEGUNDOS or duracao_s) / 60
print(f"Detecção:       {len(ident)} tracks no total, {len(validos_ident)} com tamanho suficiente")
print(f"Identificação:  {auto} de {len(validos_ident)} tracks ({auto / max(len(validos_ident), 1):.0%}) identificados pelo número, sem ajuda")
print(f"Fragmentação:   {tabela['tracks'].mean():.1f} tracks por atleta em média (perto de 1 = rastreio estável)")
print(f"Alertas:        {(tabela['alerta_identidade'] == 'sim').sum()} atletas com tracks sobrepostos (provável erro de identidade)")
print(f"Revisão:        {TEMPO_REVISAO_S / 60:.1f} min (meta: menos de 3 a 5 min por treino)")
print(f"Processamento:  {TEMPO_PROCESSAMENTO_S / 60:.1f} min para {min_video:.1f} min de vídeo "
      f"({TEMPO_PROCESSAMENTO_S / max(min_video, 0.01):.0f} s de GPU por minuto de vídeo)")
print("\nPara conferir a distância: compare a de um atleta com a de um relógio com GPS no mesmo treino (até ~15% de diferença está ok).")